<a href="https://colab.research.google.com/github/mwanafadhili21-creator/mosi/blob/main/blue_economy_rnn_lstm_gru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Blue Economy Time Series Forecasting: RNN vs LSTM vs GRU
### Indian Ocean Tuna Commission (IOTC) Catch Data — WPTT28

This notebook compares **RNN, LSTM, and GRU** architectures for forecasting monthly tropical tuna catch
(Yellowfin, Bigeye, Skipjack) in the Indian Ocean, using official IOTC catch-and-effort data.

**Run this in Google Colab** (Runtime → Change runtime type → GPU) for TensorFlow/Keras support.

Upload `monthly_total_catch.csv` and `monthly_species_catch.csv` to the Colab session (or your Drive) before running.


## 1. Setup

In [ ]:
!pip install -q tensorflow scikit-learn pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

tf.random.set_seed(42)
np.random.seed(42)


## 2. Load data
If running in Colab, upload the two CSVs first (files icon on the left → upload), or mount Drive.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # uncomment to upload interactively

total_df = pd.read_csv("monthly_total_catch.csv", parse_dates=["date"])
species_df = pd.read_csv("monthly_species_catch.csv", parse_dates=["date"])

print(total_df.shape, species_df.shape)
total_df.head()


## 3. Exploratory look at the series

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(total_df["date"], total_df["catch_mt_total"])
ax.set_title("Total Monthly Tropical Tuna Catch — Indian Ocean (MT)")
ax.set_xlabel("Year"); ax.set_ylabel("Catch (metric tons)")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(12,4))
for col in ["Yellowfin tuna","Bigeye tuna","Skipjack tuna"]:
    ax.plot(species_df["date"], species_df[col], label=col)
ax.set_title("Monthly Catch by Species (MT)")
ax.legend(); plt.tight_layout(); plt.show()


## 4. Preprocessing
We forecast **total monthly catch**. Scale to [0,1] and build sliding windows (12-month lookback → predict next month).

In [ ]:
series = total_df["catch_mt_total"].values.reshape(-1,1)

# chronological split: 70% train, 15% val, 15% test
n = len(series)
train_end = int(n*0.70)
val_end = int(n*0.85)

scaler = MinMaxScaler()
scaler.fit(series[:train_end])  # fit ONLY on train to avoid leakage
series_scaled = scaler.transform(series)

LOOKBACK = 12  # months

def make_windows(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback, 0])
        y.append(data[i+lookback, 0])
    return np.array(X), np.array(y)

X_all, y_all = make_windows(series_scaled, LOOKBACK)

# align split indices (shifted by lookback)
train_end_w = train_end - LOOKBACK
val_end_w = val_end - LOOKBACK

X_train, y_train = X_all[:train_end_w], y_all[:train_end_w]
X_val, y_val = X_all[train_end_w:val_end_w], y_all[train_end_w:val_end_w]
X_test, y_test = X_all[val_end_w:], y_all[val_end_w:]

# reshape for RNN input: (samples, timesteps, features)
X_train = X_train.reshape(-1, LOOKBACK, 1)
X_val = X_val.reshape(-1, LOOKBACK, 1)
X_test = X_test.reshape(-1, LOOKBACK, 1)

print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)


## 5. Model builders
Same layer size (32 units) and training setup for all three, so the comparison isolates the *architecture* effect.

In [ ]:
def build_model(cell_type, units=32, lookback=LOOKBACK, dropout=0.2):
    layer = {"RNN": SimpleRNN, "LSTM": LSTM, "GRU": GRU}[cell_type]
    model = Sequential([
        layer(units, activation="tanh", input_shape=(lookback, 1)),
        Dropout(dropout),
        Dense(16, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

results = {}
histories = {}
predictions = {}

for cell_type in ["RNN", "LSTM", "GRU"]:
    print(f"\n=== Training {cell_type} ===")
    model = build_model(cell_type)
    es = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
    hist = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=200, batch_size=16,
        callbacks=[es], verbose=0
    )
    histories[cell_type] = hist

    y_pred_scaled = model.predict(X_test, verbose=0).flatten()
    y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1,1)).flatten()
    y_true = scaler.inverse_transform(y_test.reshape(-1,1)).flatten()
    predictions[cell_type] = (y_true, y_pred)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)

    results[cell_type] = {
        "RMSE (MT)": rmse, "MAE (MT)": mae, "MAPE (%)": mape, "R2": r2,
        "epochs_trained": len(hist.history["loss"]),
        "params": model.count_params()
    }
    print(f"{cell_type}: RMSE={rmse:,.0f}  MAE={mae:,.0f}  MAPE={mape:.2f}%  R2={r2:.3f}")


## 6. Comparison table

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df[["RMSE (MT)","MAE (MT)","MAPE (%)","R2","epochs_trained","params"]]
results_df


## 7. Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4), sharey=True)
for ax, cell_type in zip(axes, ["RNN","LSTM","GRU"]):
    h = histories[cell_type]
    ax.plot(h.history["loss"], label="train")
    ax.plot(h.history["val_loss"], label="val")
    ax.set_title(cell_type); ax.set_xlabel("epoch"); ax.legend()
axes[0].set_ylabel("MSE loss (scaled)")
plt.tight_layout(); plt.show()


## 8. Forecasts vs actuals on the test set

In [ ]:
fig, ax = plt.subplots(figsize=(13,5))
test_dates = total_df["date"].values[-len(y_test):]
y_true_ref = predictions["LSTM"][0]
ax.plot(test_dates, y_true_ref, label="Actual", color="black", linewidth=2)
for cell_type, color in zip(["RNN","LSTM","GRU"], ["tab:red","tab:blue","tab:green"]):
    _, y_pred = predictions[cell_type]
    ax.plot(test_dates, y_pred, label=cell_type, color=color, alpha=0.8)
ax.set_title("Test Set: Actual vs Predicted Monthly Catch")
ax.set_ylabel("Catch (MT)"); ax.legend()
plt.tight_layout(); plt.show()


## 9. Discussion

Fill this section in after running, using your actual numbers:

- **Best performer (lowest RMSE/MAPE):** ...
- **Fastest to converge (fewest epochs / smallest val loss gap):** GRU and LSTM usually converge faster and more stably than vanilla RNN on this kind of seasonal, moderate-length series (~700 points).
- **Parameter count:** GRU has ~25% fewer parameters than LSTM for the same unit count, since it uses 2 gates instead of 3.
- **Practical takeaway for blue-economy forecasting:** with a dataset this size (708 monthly points), GRU/LSTM should meaningfully outperform vanilla RNN by capturing the strong seasonal (12-month) cycle in tuna catch; GRU is a reasonable default when compute/training time matters (e.g. deploying on modest hardware for fisheries monitoring), while LSTM may edge ahead if you extend the lookback window or add exogenous effort/CPUE features.

### Suggested extensions
- Add fishing effort (from `EF.csv`) as a second input feature (multivariate forecasting)
- Forecast per-species catch instead of the aggregate
- Try a longer lookback (24 months) to capture El Niño / IOD-driven multi-year cycles
- Compare against a classical baseline (SARIMA) to quantify the deep learning gain
